In [167]:
import pandas as pd

df = pd.read_csv("http://114.207.245.181:13000/csv/ynat.csv")
print(df['predefined_news_category'].value_counts())
print(df.columns)
df = df.head(1000)

predefined_news_category
세계      8034
정치      7993
IT과학    7852
스포츠     7688
경제      6028
생활문화    6022
사회      2061
Name: count, dtype: int64
Index(['guid', 'title', 'predefined_news_category', 'label', 'url', 'date',
       'annotations.annotators', 'annotations.annotations.first-scope',
       'annotations.annotations.second-scope',
       'annotations.annotations.third-scope'],
      dtype='object')


In [168]:
# None 데이터 삭제하기
df.dropna(inplace=True)
df.shape

(1000, 10)

In [169]:
mapping = {
    "세계": 0,
    "정치": 1,
    "IT과학": 2,
    "스포츠":  3,
    "경제": 4,
    "생활문화": 5,
    "사회": 6,
}

df['target'] = df['predefined_news_category'].map(mapping)
df['target']

0      2
1      5
2      2
3      2
4      5
      ..
995    0
996    4
997    3
998    1
999    2
Name: target, Length: 1000, dtype: int64

In [170]:
texts = df['title'].to_list()
labels = torch.tensor(df['target'].values, dtype=torch.long)
labels.shape

torch.Size([1000])

In [156]:
MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

In [171]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [158]:
inputs = tokenizer(
    texts, 
    padding=True, 
    truncation=True, 
    return_tensors="pt", 
    max_length=100
)
inputs


{'input_ids': tensor([[    2, 13075, 18376,  ...,     0,     0,     0],
        [    2, 32734,  5419,  ...,     0,     0,     0],
        [    2, 19897,  8558,  ...,     0,     0,     0],
        ...,
        [    2,  3553, 27771,  ...,     0,     0,     0],
        [    2,  9827,  9235,  ...,     0,     0,     0],
        [    2, 18156,  5223,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [172]:
# 2. Dataset 생성
import torch
from torch.utils.data import DataLoader, TensorDataset

# input_ids, attention_mask, labels로 구성
dataset = TensorDataset(
    inputs['input_ids'],
    inputs['attention_mask'],
    labels,
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [173]:
# 모델 생성

import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

class BertClassifier(nn.Module):
    
    # 레이어 생성
    def __init__(self):
        super().__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        for param in self.bert.parameters():
            param.requires_grad = False
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(hidden_size, 7)

    # 실행
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids = input_ids,
            attention_mask = attention_mask
        )
        cls = outputs['last_hidden_state'][:, 0]
        x = self.dropout(cls)
        x = self.fc1(x)
        return x


In [174]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BertClassifier().to(device)

# binary_crossentropy in tensorflow
# criterion = nn.BCEWithLogitsLoss()
criterion = nn.CrossEntropyLoss()

# L2규제(?)
# filter를 통해서 parameters중에서 requires_grad = False는 제거
# learning_rate는 0.001
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr = 0.001
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [175]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    correct = 0
    total = 0

    for batch in loader:
        # NOTE: this batch[indexing] is not related to loader(batch_size=n)
        # it's from TensorDataset(...)
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()                                                 # 기울기 초기화
        logits = model(input_ids = input_ids,attention_mask = attention_mask) # 모델에 학습
        loss = criterion(logits.squeeze(), labels.long())                     # 로그 계산
        loss.backward()                                                       # 기울기 계산
        optimizer.step()                                                      # 가중치 업데이트
        total_loss += loss.item()                                             # 전체 loss값 누적

        # 추가된 부'bi
        preds = torch.argmax(logits, dim=1)
        correct += ( preds == labels).sum().item()

        total += labels.size(0)

    accuracy = correct / total
    # epoch와 loss값 출력 accuracy 출력
    print(f"Epoch {epoch + 1} Loss {total_loss/len(loader):.4f}, Accuracy: {accuracy:.4f}")

Epoch 1 Loss 0.9492, Accuracy: 0.6860
Epoch 2 Loss 0.4707, Accuracy: 0.8570
Epoch 3 Loss 0.3688, Accuracy: 0.8850
Epoch 4 Loss 0.3350, Accuracy: 0.8840
Epoch 5 Loss 0.2815, Accuracy: 0.9190
Epoch 6 Loss 0.2610, Accuracy: 0.9140
Epoch 7 Loss 0.2352, Accuracy: 0.9240
Epoch 8 Loss 0.2301, Accuracy: 0.9210
Epoch 9 Loss 0.2257, Accuracy: 0.9200
Epoch 10 Loss 0.2174, Accuracy: 0.9370


In [165]:
model.eval()

BertClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(40000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, ele

In [ ]:
test_texts = [
    "이번년도 물가 상승률이 급상승 할 것으로 예상됩니다. ",
    "Apple에서 새로운 AI 모델 출시 ",
    "월드컵 조 추첨 결과 발표",
    "6월 3일 지방선거 결과",
    "미 백악관에서 트럼프 80세 생일날 옥타곤 이벤트",
    "가족센터서 임종·장례 가족 심리 지원…가정폭력 보호 1년6개월까지",
    "괴산 감물 감자 맛보세요 12∼14일 감물면서 축제",
]

for test_text in test_texts:
    # 1. 토크나이저
    encoding = tokenizer(
        test_text, 
        padding=True, 
        truncation=True, 
        return_tensors="pt", 
        max_length=100
    )

    with torch.no_grad():
        logits = model(
            encoding['input_ids'].to(device),
            encoding['attention_mask'].to(device)
        )

        # 3. Apply Softmax along the class dimension (dim=-1)
        prob = torch.softmax(logits, dim=-1)

    # Print raw probabilities and the final predicted class index
    category_names = list(mapping.keys())
    most_likely_index = torch.argmax(prob).item()
    print(f"뉴스 타이틀: {test_text}", end="=> ")
    print(f"뉴스분류 예측 결과 (클래스 인덱스): {category_names[most_likely_index]}({most_likely_index})")
    print(" | ".join([f"{category_names[i]:}: {p:.3f}" for i, p in enumerate(prob[0])]))
    print("")
    

뉴스 타이틀: 이번년도 물가 상승률이 급상승 할 것으로 예상됩니다. => 뉴스분류 예측 결과 (클래스 인덱스): 경제(4)
세계: 0.001 | 정치: 0.015 | IT과학: 0.026 | 스포츠: 0.001 | 경제: 0.775 | 생활문화: 0.174 | 사회: 0.010

뉴스 타이틀: Apple에서 새로운 AI 모델 출시 => 뉴스분류 예측 결과 (클래스 인덱스): IT과학(2)
세계: 0.001 | 정치: 0.000 | IT과학: 0.999 | 스포츠: 0.000 | 경제: 0.000 | 생활문화: 0.000 | 사회: 0.000

뉴스 타이틀: 월드컵 조 추첨 결과 발표=> 뉴스분류 예측 결과 (클래스 인덱스): 스포츠(3)
세계: 0.004 | 정치: 0.003 | IT과학: 0.000 | 스포츠: 0.992 | 경제: 0.000 | 생활문화: 0.001 | 사회: 0.000

뉴스 타이틀: 6월 3일 지방선거 결과=> 뉴스분류 예측 결과 (클래스 인덱스): 정치(1)
세계: 0.000 | 정치: 0.983 | IT과학: 0.000 | 스포츠: 0.000 | 경제: 0.009 | 생활문화: 0.005 | 사회: 0.003

뉴스 타이틀: 미 백악관에서 트럼프 80세 생일날 옥타곤 이벤트=> 뉴스분류 예측 결과 (클래스 인덱스): 세계(0)
세계: 0.993 | 정치: 0.006 | IT과학: 0.000 | 스포츠: 0.000 | 경제: 0.000 | 생활문화: 0.000 | 사회: 0.000

뉴스 타이틀: 가족센터서 임종·장례 가족 심리 지원…가정폭력 보호 1년6개월까지=> 뉴스분류 예측 결과 (클래스 인덱스): 세계(0)
세계: 0.410 | 정치: 0.184 | IT과학: 0.020 | 스포츠: 0.000 | 경제: 0.308 | 생활문화: 0.005 | 사회: 0.074

뉴스 타이틀: 괴산 감물 감자 맛보세요 12∼14일 감물면서 축제=> 뉴스분류 예측 결과 (클래스 인덱스): 생활문화(5)
세계: 0.004 | 정치: 0.004 | I

In [ ]:
print(list(mapping.keys())[i])

세계
